In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, r2_score
from sklearn import linear_model
import numpy as np
from sklearn.preprocessing import OneHotEncoder
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
import ast
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
import locale

Data Cleaning

In [2]:
#Merge company data and individual data 
df_company = pd.read_csv('dealroom_data_by_company.csv')
df_person = pd.read_csv('Dealroom_data_by_person.csv')

merged=pd.merge(df_company, df_person, on='Link')

#Clean data by removing and renaming columns
columns= ['Company name_y', 'Founders data']
merged = merged.drop(columns, axis=1)
merged=merged.rename(columns={"Company name_x": "Company_Name"})
display(merged)

,Link,Company_Name,Industries,Funding,Name,Title,Gender
0,https://app.dealroom.co/companies/uptrek,IJsfontein,"['gaming', 'education']",Unknown,Thorsten Unger,CEO& Investor,male
1,https://app.dealroom.co/companies/uptrek,IJsfontein,"['gaming', 'education']",Unknown,Guido Boogaard,CEO& Investor,male
2,https://app.dealroom.co/companies/uptrek,IJsfontein,"['gaming', 'education']",Unknown,Hermann Kudlich,Advisor,male
3,https://app.dealroom.co/companies/uptrek,IJsfontein,"['gaming', 'education']",Unknown,Marvin van Kalsbeek,Co-Founder,male
4,https://app.dealroom.co/companies/uptrek,IJsfontein,"['gaming', 'education']",Unknown,Hayo Wagenaar,Co-Founder,male
...,...,...,...,...,...,...,...
723,https://app.dealroom.co/companies/songvice,Songvice,"['music', 'education', 'music']",Unknown,Tiago Martins,CTO& Co-Founder,male
724,https://app.dealroom.co/companies/songvice,Songvice,"['music', 'education', 'music']",Unknown,Pedro Silva,CTO& Co-Founder,male
725,https://app.dealroom.co/companies/songvice,Songvice,"['music', 'education', 'music']",Unknown,Luís Dias,CEO,male
726,https://app.dealroom.co/companies/songvice,Songvice,"['music', 'education', 'music']",Unknown,Barney Graman,Co-founder & Content Director,male


Create Linear Regression Model

In [3]:
#Select rows that have a value for funding
filtered_funds = merged[merged['Funding'] !='Unknown']
#display(filtered_funds)

#Strip values of brackets
filtered_funds.loc[:,'Funding'] = filtered_funds['Funding'].astype(str).str.strip("[]")
#display(filtered_funds)

#Clean funding values
filtered_funds.loc[:,'Funding']=[value.replace('$', '').replace('.', '').replace('k', '000').replace('m', '000000').strip("'") for value in filtered_funds['Funding']]


#Get rid of empty strings and convert to float
filtered_funds=filtered_funds.loc[filtered_funds['Funding'].str.len()>0]
filtered_funds['Funding']=filtered_funds['Funding'].astype(float)
#display(filtered_funds)

#strip of quotation marks around industries
filtered_funds['Industries']=filtered_funds['Industries'].apply(ast.literal_eval)
#display(filtered_funds)
  

In [4]:
#Remove outliers to improve score
threshold=filtered_funds['Funding'].quantile(0.95)
filtered_funds=filtered_funds[filtered_funds['Funding']<=threshold]
display(filtered_funds)

,Link,Company_Name,Industries,Funding,Name,Title,Gender
13,https://app.dealroom.co/companies/welectric_,Welectric,[transportation],550000.0,Wouter Klaase,Co-Founder,male
14,https://app.dealroom.co/companies/welectric_,Welectric,[transportation],550000.0,Ivo Brandsma,Co-Founder,male
17,https://app.dealroom.co/companies/uni_life,Uni-Life,"[education, marketing]",719000.0,Joep Annega,Founder,male
18,https://app.dealroom.co/companies/uni_life,Uni-Life,"[education, marketing]",719000.0,Thomas Smulders,Founder,male
19,https://app.dealroom.co/companies/uni_life,Uni-Life,"[education, marketing]",719000.0,Joep Annega,Co-Founder& COO,male
...,...,...,...,...,...,...,...
695,https://app.dealroom.co/companies/vood_formerl...,Vood (Formerly Plant B),[food],275000.0,Hein de Jong,Former Co-Founder,male
702,https://app.dealroom.co/companies/ozarka_b_v_,Ozarka,"[food, energy]",770000.0,Michael Massa,Co-Founder,male
703,https://app.dealroom.co/companies/ozarka_b_v_,Ozarka,"[food, energy]",770000.0,Beth Massa,Co-Founder,female
713,https://app.dealroom.co/companies/vylo,Vylo,[media],20000000.0,Tyler Reynolds,CEO& Co-Founder,male


In [5]:
#One Hot Encode Gender
column=filtered_funds['Gender']
values= np.array(column).reshape(-1,1)
encoding= OneHotEncoder(sparse=False)
gender= pd.DataFrame(encoding.fit_transform(values))
columns = np.unique(filtered_funds[ 'Gender'])
columns = list(columns)
gender.columns=columns
#display(gender)

c:\Users\majak\anaconda3\lib\site-packages\sklearn\preprocessing\_encoders.py:972: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


In [6]:
#One Hot Encoding Industry - for all values in list
from sklearn.preprocessing import MultiLabelBinarizer
mlb = MultiLabelBinarizer()
industry = pd.DataFrame(mlb.fit_transform(filtered_funds['Industries']), columns=mlb.classes_, index=filtered_funds.index)
#display(industry)

In [7]:
#Reset index and merge gender with industries to have them as features for model
gender.reset_index(drop=True, inplace=True)
industry.reset_index(drop=True, inplace=True)
all_features=pd.concat([gender, industry], axis=1)
display(all_features)

,female,male,dating,education,energy,enterprise software,event tech,fashion,fintech,food,...,legal,marketing,media,music,real estate,robotics,security,transportation,travel,wellness beauty
0,0.0,1.0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
1,0.0,1.0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
2,0.0,1.0,0,1,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
3,0.0,1.0,0,1,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
4,0.0,1.0,0,1,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
350,0.0,1.0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
351,0.0,1.0,0,0,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
352,1.0,0.0,0,0,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
353,0.0,1.0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0


In [8]:
#Scale funding values to improve model
X= all_features
y=filtered_funds['Funding']

scaler = StandardScaler()
y = scaler.fit_transform(y.values.reshape(-1, 1))


In [9]:
# Split the data for training and testing
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# Linear regression model
linear_model = LinearRegression()

# Train data
linear_model.fit(X_train, y_train)

# Get predictions
y_predictions = linear_model.predict(X_test)

#Reverse standardization to get ture valeus for fuding
y_transformed = scaler.inverse_transform(y_predictions.reshape(-1, 1))
#print(y_transformed)

# Check model accuracy
mse = mean_absolute_error(y_test, y_predictions)
print(f'Mean Squared Error: {mse}')



Mean Squared Error: 0.37264044964659343


Predicting Funding on Unknown Values

In [10]:
#Select funding values that are unkown from dataframe
predictions = merged[merged['Funding'] =='Unknown']
predictions['Industries']=predictions['Industries'].apply(ast.literal_eval)

C:\Users\majak\AppData\Local\Temp\ipykernel_7488\1349087051.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  predictions['Industries']=predictions['Industries'].apply(ast.literal_eval)


In [11]:
#Repeat preprocessing of data

#One Hot Encoding for industries
from sklearn.preprocessing import MultiLabelBinarizer
mlb = MultiLabelBinarizer()
industry_predictions = pd.DataFrame(mlb.fit_transform(filtered_funds['Industries']), columns=mlb.classes_, index=filtered_funds.index)

#Use One Hot Encoder for categorical data - Gender
predictions.loc[:,'Gender'] = predictions['Gender'].astype(str)
column=predictions['Gender']
gv= np.array(column).reshape(-1,1)
encoding= OneHotEncoder(sparse=False)
gender_predictions= pd.DataFrame(encoding.fit_transform(gv))
columns = np.unique(predictions['Gender'])
columns = list(columns)
gender_predictions.columns=columns
#display(gender_predictions)

#Merge dataframes to create X
gender_predictions.reset_index(drop=True, inplace=True)
industry_predictions.reset_index(drop=True, inplace=True)
X2=pd.concat([gender, industry], axis=1)
X2=pd.merge(gender_predictions, industry_predictions, left_index=True, right_index=True, how='left')


X2 = X2.drop(columns='nan')
display(X2)

c:\Users\majak\anaconda3\lib\site-packages\sklearn\preprocessing\_encoders.py:972: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


,female,male,dating,education,energy,enterprise software,event tech,fashion,fintech,food,...,legal,marketing,media,music,real estate,robotics,security,transportation,travel,wellness beauty
0,0.0,1.0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
1,0.0,1.0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
2,0.0,1.0,0,1,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
3,0.0,1.0,0,1,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
4,0.0,1.0,0,1,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
349,0.0,1.0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
350,0.0,1.0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
351,0.0,1.0,0,0,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
352,0.0,1.0,0,0,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


In [12]:
# Predict unknwon values using model
unknwon_funding = linear_model.predict(X2)

#Reverse standardization
pred = scaler.inverse_transform(unknwon_funding.reshape(-1, 1))
print(pred)

[[ 1.26082383e+07]
 [ 1.26082383e+07]
 [ 3.58957122e+07]
 [ 3.58957122e+07]
 [ 3.58957122e+07]
 [ 2.54150401e+07]
 [ 2.54150401e+07]
 [ 2.54150401e+07]
 [ 2.21887199e+07]
 [ 2.54150401e+07]
 [ 2.54150401e+07]
 [ 2.54150401e+07]
 [ 2.21887199e+07]
 [ 2.21887199e+07]
 [ 2.21887199e+07]
 [ 4.51846601e+07]
 [ 4.19583399e+07]
 [ 4.19583399e+07]
 [ 4.19583399e+07]
 [ 2.88903976e+07]
 [ 2.88903976e+07]
 [ 2.88903976e+07]
 [ 2.88903976e+07]
 [ 1.08886165e+07]
 [ 1.08886165e+07]
 [ 7.66229632e+06]
 [ 1.08886165e+07]
 [ 2.40191597e+07]
 [ 2.40191597e+07]
 [ 4.70843154e+07]
 [ 4.70843154e+07]
 [ 1.08886165e+07]
 [ 1.13738445e+07]
 [ 8.14752430e+06]
 [ 1.13738445e+07]
 [ 1.13738445e+07]
 [ 4.70843154e+07]
 [ 4.70843154e+07]
 [ 4.70843154e+07]
 [ 4.70843154e+07]
 [ 1.76118485e+07]
 [ 1.76118485e+07]
 [ 1.76118485e+07]
 [ 2.72094860e+07]
 [ 2.72094860e+07]
 [ 2.72094860e+07]
 [ 2.72094860e+07]
 [ 2.72094860e+07]
 [ 2.26204615e+07]
 [ 2.26204615e+07]
 [ 2.58467817e+07]
 [ 2.58467817e+07]
 [ 3.0763445

In [13]:
#Rewrite funding values to make them more readable

# Use locale system
locale.setlocale(locale.LC_ALL, '')

#Get predictions to be in an array
pred_ar = np.array(pred)

# Format without the dollar sign
def format_number(amount):
    return locale.format_string('%d', amount, grouping=True)

# Apply function to all values
final_predictions = np.vectorize(format_number)(pred_ar)
print(final_predictions)

[['12.608.238']
 ['12.608.238']
 ['35.895.712']
 ['35.895.712']
 ['35.895.712']
 ['25.415.040']
 ['25.415.040']
 ['25.415.040']
 ['22.188.719']
 ['25.415.040']
 ['25.415.040']
 ['25.415.040']
 ['22.188.719']
 ['22.188.719']
 ['22.188.719']
 ['45.184.660']
 ['41.958.339']
 ['41.958.339']
 ['41.958.339']
 ['28.890.397']
 ['28.890.397']
 ['28.890.397']
 ['28.890.397']
 ['10.888.616']
 ['10.888.616']
 ['7.662.296']
 ['10.888.616']
 ['24.019.159']
 ['24.019.159']
 ['47.084.315']
 ['47.084.315']
 ['10.888.616']
 ['11.373.844']
 ['8.147.524']
 ['11.373.844']
 ['11.373.844']
 ['47.084.315']
 ['47.084.315']
 ['47.084.315']
 ['47.084.315']
 ['17.611.848']
 ['17.611.848']
 ['17.611.848']
 ['27.209.485']
 ['27.209.485']
 ['27.209.485']
 ['27.209.485']
 ['27.209.485']
 ['22.620.461']
 ['22.620.461']
 ['25.846.781']
 ['25.846.781']
 ['30.763.445']
 ['30.763.445']
 ['30.763.445']
 ['30.763.445']
 ['30.763.445']
 ['30.763.445']
 ['30.763.445']
 ['30.763.445']
 ['-6.078.078']
 ['-9.304.399']
 ['22.620.

In [14]:
#Display results in dataframe
predictions.loc[:,'Predicted Funding'] = final_predictions
predictions = predictions.drop('Funding', axis=1)
display(predictions)

C:\Users\majak\AppData\Local\Temp\ipykernel_7488\1328749582.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  predictions.loc[:,'Predicted Funding'] = final_predictions


,Link,Company_Name,Industries,Name,Title,Gender,Predicted Funding
0,https://app.dealroom.co/companies/uptrek,IJsfontein,"[gaming, education]",Thorsten Unger,CEO& Investor,male,12.608.238
1,https://app.dealroom.co/companies/uptrek,IJsfontein,"[gaming, education]",Guido Boogaard,CEO& Investor,male,12.608.238
2,https://app.dealroom.co/companies/uptrek,IJsfontein,"[gaming, education]",Hermann Kudlich,Advisor,male,35.895.712
3,https://app.dealroom.co/companies/uptrek,IJsfontein,"[gaming, education]",Marvin van Kalsbeek,Co-Founder,male,35.895.712
4,https://app.dealroom.co/companies/uptrek,IJsfontein,"[gaming, education]",Hayo Wagenaar,Co-Founder,male,35.895.712
...,...,...,...,...,...,...,...
723,https://app.dealroom.co/companies/songvice,Songvice,"[music, education, music]",Tiago Martins,CTO& Co-Founder,male,10.888.616
724,https://app.dealroom.co/companies/songvice,Songvice,"[music, education, music]",Pedro Silva,CTO& Co-Founder,male,10.888.616
725,https://app.dealroom.co/companies/songvice,Songvice,"[music, education, music]",Luís Dias,CEO,male,9.978.476
726,https://app.dealroom.co/companies/songvice,Songvice,"[music, education, music]",Barney Graman,Co-founder & Content Director,male,9.978.476


In [16]:
predictions['Predicted Funding'] = predictions['Predicted Funding'].str.replace(',', '')
predictions['Predicted Funding'] = predictions['Predicted Funding'].str.replace('.', '')
predictions['Predicted Funding'] = predictions['Predicted Funding'].astype(float)
average_funding = predictions.groupby('Gender')['Predicted Funding'].mean()

numeric = average_funding[average_funding.notna()]

#Get values
average_female=numeric[0]
average_male=numeric[1]

#Format values
f_female = locale.format_string('%.2f', average_female, grouping=True)
f_male = locale.format_string('%.2f', average_male, grouping=True)


print(f"Average predicted funding for women: {f_female}")
print(f"Average predicted funding for men: {f_male}")


Average predicted funding for women: 45.058.395,33
Average predicted funding for men: 47.650.447,89


C:\Users\majak\AppData\Local\Temp\ipykernel_7488\2986055281.py:9: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  average_female=numeric[0]
C:\Users\majak\AppData\Local\Temp\ipykernel_7488\2986055281.py:10: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  average_male=numeric[1]


In [17]:
#predictions.to_csv('correct_funding_predictions.csv', index=False)
import pandas as pd

predictions['Average predicted funding women']= pd.Series(f_female)
predictions['Average predicted funding men']= pd.Series(f_male)
with open('correct_funding_predictions.csv', 'w', newline='') as file:
    predictions.to_csv('correct_funding_predictions.csv', index=False)

Linear Regression Model Summary

In [23]:
# Residuals
residuals = y_test - y_predictions

# Degrees of freedom
n = len(y_test)
p = X_test.shape[1]  # number of features
df_residuals = n - p - 1
df_total = n - 1

# Mean squared error
mse = mean_absolute_error(y_test, y_predictions)

# R-squared
ssr = np.sum((y_predictions - np.mean(y_test))**2)
sst = np.sum((y_test - np.mean(y_test))**2)
r_squared = r2_score(y_test, y_predictions)

# Adjusted R-squared
adjusted_r_squared = 1 - (1 - r_squared) * (n - 1) / df_residuals

# F-statistic
f_statistic = (ssr / p) / (mse / df_residuals)

# Significance level (p-value) for F-statistic
from scipy.stats import f
p_value_f = 1 - f.cdf(f_statistic, p, df_residuals)

# Displaying the results
print(f'Mean Squared Error (MSE): {mse}')
print(f'R-squared: {r_squared}')
print(f'Adjusted R-squared: {adjusted_r_squared}')
print(f'F-statistic: {f_statistic}')
print(f'Significance Level (p-value) for F-statistic: {p_value_f}')

Mean Squared Error (MSE): 0.37264044964659343
R-squared: 0.7226440927159126
Adjusted R-squared: 0.6245027716769278
F-statistic: 638.5176604931761
Significance Level (p-value) for F-statistic: 1.1102230246251565e-16


In [27]:
coefficients = linear_model.coef_

# Ensure coefficients is one-dimensional
coefficients = coefficients.ravel()

# Create a DataFrame to store feature names and coefficients
coefficients_df = pd.DataFrame({'Feature': X_train.columns, 'Coefficient': coefficients})

# Displaying the coefficients
print(coefficients_df)

                Feature  Coefficient
0                female    -0.025010
1                  male     0.025010
2                dating    -0.414111
3             education     0.283463
4                energy    -0.014110
5   enterprise software     0.308128
6            event tech     0.373209
7               fashion     0.264567
8               fintech    -0.343365
9                  food    -0.267140
10               health    -0.263042
11          home living    -0.287363
12     jobs recruitment    -0.296179
13                legal    -0.670479
14            marketing    -0.162907
15                media    -0.096711
16                music    -0.434091
17          real estate     0.301435
18             robotics    -0.350241
19             security     2.791669
20       transportation    -0.240480
21               travel     0.826204
22      wellness beauty    -0.063571
